In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import scipy.sparse
import sys
import os

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('.'))))

from src.utils import train_and_evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("="*50)
print("STEP 8-12: EVALUATION & RESULTS")
print("="*50)

In [ ]:
# Load features
print("Loading features...")
X_train_tfidf = scipy.sparse.load_npz('data/X_train_tfidf.npz')
X_test_tfidf = scipy.sparse.load_npz('data/X_test_tfidf.npz')
y_train = np.load('data/y_train.npy')
y_test = np.load('data/y_test.npy')

print(f"✅ Features loaded!")

In [ ]:
# Train all models
results = []

print("\n🔹 Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
results.append(train_and_evaluate(lr, "Logistic Regression", X_train_tfidf, X_test_tfidf, y_train, y_test))

print("\n🔹 Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
results.append(train_and_evaluate(rf, "Random Forest", X_train_tfidf, X_test_tfidf, y_train, y_test))

print("\n🔹 Training SVM...")
svm = SVC(kernel='linear', probability=True, random_state=42)
results.append(train_and_evaluate(svm, "SVM", X_train_tfidf, X_test_tfidf, y_train, y_test))

print("\n🔹 Training XGBoost...")
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
results.append(train_and_evaluate(xgb, "XGBoost", X_train_tfidf, X_test_tfidf, y_train, y_test))

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4)

print("\n" + "="*60)
print("📊 MODEL COMPARISON SUMMARY")
print("="*60)
print(comparison_df.to_string(index=False))

# Find best model
best = comparison_df.loc[comparison_df['accuracy'].idxmax()]
print(f"\n🏆 BEST MODEL: {best['model']} with Accuracy: {best['accuracy']:.4f}")

In [ ]:
# Save results for partner (Atif)
print("\n💾 Saving results for Atif's visualizations...")

# Save comparison table
comparison_df.to_csv('results/model_comparison.csv', index=False)
print("✅ results/model_comparison.csv")

# Save confusion matrices
confusion_data = [{'model': r['model'], 'cm': r['confusion_matrix'].tolist()} for r in results]
with open('results/confusion_matrices.json', 'w') as f:
    json.dump(confusion_data, f)
print("✅ results/confusion_matrices.json")

# Save predictions and probabilities for ROC curves
for r in results:
    name = r['model'].replace(" ", "_").lower()
    np.save(f'results/{name}_predictions.npy', r['y_pred'])
    if r['y_pred_proba'] is not None:
        np.save(f'results/{name}_probabilities.npy', r['y_pred_proba'])
    print(f"✅ results/{name}_predictions.npy")

In [ ]:
# Save best model
joblib.dump(xgb, 'models/best_model_xgboost.pkl')
print("✅ models/best_model_xgboost.pkl")

In [ ]:
# Test with new messages
from src.utils import clean_text

# Load vectorizer
vectorizer = joblib.load('models/tfidf_vectorizer.pkl')

def predict_spam(message, model, vec):
    cleaned = clean_text(message)
    transformed = vec.transform([cleaned])
    pred = model.predict(transformed)[0]
    prob = model.predict_proba(transformed)[0][1]
    result = "SPAM" if pred == 1 else "HAM"
    confidence = prob if pred == 1 else 1 - prob
    return result, confidence

# Test messages
test_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Verify now!",
    "Can you please send me the assignment by evening?"
]

print("\n" + "="*50)
print("🔍 TESTING MODEL WITH NEW MESSAGES")
print("="*50)

for msg in test_messages:
    result, confidence = predict_spam(msg, xgb, vectorizer)
    print(f"\n📧 Message: {msg}
    )
    print(f"📊 Prediction: {result} (Confidence: {confidence:.2%})")

In [ ]:
# Summary
print("\n" + "="*60)
print("✅ TANZEELA'S TASKS - COMPLETED")
print("="*60)
print("""
1. ✅ Data loaded automatically (no manual download)
2. ✅ Text preprocessing completed
3. ✅ TF-IDF feature engineering
4. ✅ Train-test split (80-20)
5. ✅ 4 Models trained:
   - Logistic Regression
   - Random Forest  
   - SVM
   - XGBoost
6. ✅ Model evaluation completed
7. ✅ Results saved for Atif's visualizations
""")